In [ ]:
import torch

import numpy as np
import matplotlib.pyplot as plt
!pip uninstall -y torchao
!pip install torch transformers datasets peft accelerate

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    TrainerCallback
)

from peft import get_peft_model, LoraConfig, TaskType

# ==============================
# 1. LOAD DATASET
# ==============================
dataset = load_dataset("imdb")

# Reduce size for faster testing
dataset["train"] = dataset["train"].select(range(2000))
dataset["test"] = dataset["test"].select(range(500))

# ==============================
# 2. TOKENIZATION
# ==============================
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(example):
    return tokenizer(example["text"], truncation=True)

dataset = dataset.map(tokenize, batched=True)

# Rename label column for Trainer
dataset = dataset.rename_column("label", "labels")

# Set format
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ==============================
# 3. MODEL + LORA
# ==============================
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["query", "value"],
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(model, lora_config)

# ==============================
# 4. DYNAMIC RANK PRUNING CALLBACK
# ==============================
class DynamicRankPruningCallback(TrainerCallback):
    def __init__(self, threshold=0.01):
        self.threshold = threshold
        print("✅ Dynamic LoRA Callback Initialized")

    def on_epoch_end(self, args, state, control, **kwargs):
        model = kwargs["model"]
        print(f"\n🔍 Epoch {state.epoch} - Checking LoRA importance")

        for name, module in model.named_modules():
            if "lora_A" in name:
                importance = module.weight.abs().sum().item()

                print(f"{name}: {importance:.6f}")

                if importance < self.threshold:
                    print(f"❌ Pruning {name}")
                    with torch.no_grad():
                        module.weight.zero_()

# ==============================
# 5. TRAINING SETUP
# ==============================
training_args = TrainingArguments(
    output_dir="./results_dynamic_lora",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_steps=50,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    callbacks=[DynamicRankPruningCallback(threshold=0.005)]
)

# ==============================
# 6. TRAIN
# ==============================
print("\n🚀 Starting training...\n")
trainer.train()
print("\n✅ Training complete!\n")

# ==============================
# 7. VISUALIZATION
# ==============================
def plot_rank_distribution(model):
    importances = []
    names = []

    for name, module in model.named_modules():
        if "lora_A" in name:
            importances.append(module.weight.abs().sum().item())
            names.append(name)

    plt.figure(figsize=(12, 5))
    plt.bar(range(len(importances)), importances)
    plt.xticks(range(len(names)), names, rotation=90)
    plt.title("LoRA Layer Importance After Training")
    plt.tight_layout()
    plt.show()

plot_rank_distribution(model)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Dynamic LoRA Callback Initialized

🚀 Starting training...



Epoch,Training Loss,Validation Loss
